# Session 4 - Lab: the matching game, and the edits that decide what the code forgets

**DAT09-06 Representation Learning · Monday 16 November 2026 · 1 h 20**

Last Monday and on Friday the network redrew pixels, and the description of the
product was a side effect. Today there is no decoder. You make **two shots** of
every product, play the **matching game** on them, and find out that the list
of **edits** you chose decides what the code keeps - and what it forgets.

### The words of the lecture, and the names your AI will use

| In the lecture | In papers and in code |
|---|---|
| **two shots** of the same product | a *positive pair*, two *views* |
| an **edit**: a change to the photo that does not change the product | an *augmentation* |
| **how alike** two codes are, between −1 and 1 | the *cosine similarity* |
| the **matching game**: each shot A must pick its own shot B | the *contrastive loss*: *InfoNCE*, *NT-Xent* |
| the **grid**: every shot A against every shot B | the *similarity matrix* |
| **the other products of the batch** | the *negatives* |
| **temperature** τ: how sharp the choice must be | `temperature`, `tau` |
| **everything on the same point** | *collapse* |

### How this notebook is marked

| Marker | What it means |
|---|---|
| `# Given - read it, run it, move on.` | nothing to write. Run the cell. |
| ✏️ **YOUR TURN** | a boxed banner tells you **how many lines** and **what they do**. That is the only place you write code. |
| ❓ **Checkpoint** | **three questions** about *your* output. Answer **in writing**, in the cell below. Most only ask you to read what you just printed. |
| ⭐ **Going further** | optional, harder. Only if you are ahead: the next checkpoint never needs them. |

There are **8 cells to write**, none longer than four lines. Everything else is
given, including the training loop and every plot.

### Read this before you start

An AI will write all eight in two minutes. Let it - but **read what it writes**:
the line everyone writes first for the loss does not crash, the loss goes down,
last Monday's score does not move - and the code is three times worse at
today's job. You will meet it on purpose in checkpoint 3.

The questions are the lab. The code is the easy part.

### Six checkpoints

| | What you produce | ~ |
|---|---|---|
| **1** | Two shots of the same product: the edits, and why pixels cannot pair them | 15 min |
| **2** | The matching game: the grid, the loss, a trained encoder | 20 min |
| **3** | Did it work? Re-shot products found again - and the trap | 15 min |
| **4** | **The edits are the instruction** - four lists of edits, one table | 20 min |
| **5** | Left alone, the network cheats | 10 min |
| **6** | **The challenge** - the customers use their phone | open |

**Most of you will not reach checkpoint 6. That is deliberate.**

Run the cells **in order**.

---

## 0 · Setup

Three given cells: imports, data, and last Monday's autoencoder for comparison.

In [ ]:
# Given - read it, run it, move on.
import logging
import sys
import time
from pathlib import Path

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # a missing font is not news

sys.path.insert(0, str(Path.cwd().parent))   # make `common` importable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression

from common.evaluation import evaluate_representation
from common.paths import PROCESSED, require
from common.plotting import PALETTE, use_course_style
from common.seeding import SEED, cpu_threads, set_seed

use_course_style()
set_seed(SEED)
cpu_threads()

print("torch", torch.__version__, "| device: cpu")

### 0.2 · The catalogue

Same split as last Monday and Friday: 54,000 products to train on, 6,000 held
out. Labels are loaded, and used **only** to measure.

In [ ]:
# Given - read it, run it, move on.
blob = np.load(require(PROCESSED / "fmnist_train.npz",
                       hint="python data/prepare_data.py --fmnist"))
images, labels = blob["images"], blob["labels"]
class_names = list(blob["classes"])

x = images.reshape(len(images), -1).astype(np.float32) / 255.0   # (60000, 784)
x_train = torch.from_numpy(x[:54000].copy())
x_val = torch.from_numpy(x[54000:].copy())
y_train, y_val = labels[:54000], labels[54000:]

print(f"train {tuple(x_train.shape)}   val {tuple(x_val.shape)}")


def show_row(imgs, title=None, n=8, labels=None):
    """Plot n images side by side."""
    fig, axes = plt.subplots(1, n, figsize=(n * 1.1, 1.5))
    for c, ax in enumerate(axes):
        ax.imshow(1 - np.asarray(imgs[c]).reshape(28, 28), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if labels is not None:
            ax.set_title(labels[c], fontsize=9)
    if title:
        fig.suptitle(title, y=1.15 if labels is not None else 1.08)
    plt.show()

### 0.3 · Last Monday's autoencoder, for comparison

The exact model of last Monday: a code of 32 numbers, 8 epochs. A few seconds
on the reference machine; allow up to a minute on a laptop.

`knn` and `probe` are last Monday's two numbers: do the codes say what **kind**
of product it is?

In [ ]:
# Given - read it, run it, move on.
HIDDEN = 256


class Autoencoder(nn.Module):
    """Last Monday's model: 784 -> 256 -> d -> 256 -> 784."""

    def __init__(self, d, hidden=HIDDEN):
        super().__init__()
        self.d = d
        self.encoder = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, d))
        self.decoder = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 784), nn.Sigmoid())

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def train_ae(model, data, epochs=8, batch=256):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(data))
        for i in range(0, len(data), batch):
            xb = data[perm[i:i + batch]]
            loss = F.mse_loss(model(xb)[0], xb)
            opt.zero_grad(); loss.backward(); opt.step()


@torch.no_grad()
def codes(enc, photos):
    """The codes of some photos, as a tensor (n, d). `enc` is any encoder."""
    return enc(torch.as_tensor(photos))


set_seed(SEED)
ae32 = Autoencoder(32)
t0 = time.time()
train_ae(ae32, x_train)
ae = ae32.encoder                 # all we keep of it: its encoder
ae_rep = evaluate_representation(codes(ae, x_val).numpy(), y_val, name="AE d=32", verbose=False)
print(f"trained in {time.time() - t0:.1f}s   knn {ae_rep.knn_accuracy:.3f}   "
      f"probe {ae_rep.probe_accuracy:.3f}")

---

# Checkpoint 1 - Two shots of the same product

**~15 minutes.** Two cells to write, two lines each.

The game needs two different photos of every product, and we have one. So we
make the second one ourselves, with **edits**: changes to the photo that do not
change the product.

### 1.1 · Two edits, given

`shift` moves each photo by up to 3 pixels, up or down and left or right.
`mirror` flips half of the photos left to right (we keep it for checkpoint 4).
Both draw their own random numbers **for each photo**.

In [ ]:
# Given - read it, run it, move on.
def shift(x, px=3):
    """Edit: each photo moved by up to px pixels, up or down and left or right."""
    n = len(x)
    img = F.pad(x.view(n, 28, 28), (px, px, px, px))
    dy = torch.randint(0, 2 * px + 1, (n, 1, 1))
    dx = torch.randint(0, 2 * px + 1, (n, 1, 1))
    rows = dy + torch.arange(28).view(1, 28, 1)
    cols = dx + torch.arange(28).view(1, 1, 28)
    return img[torch.arange(n).view(n, 1, 1), rows, cols].reshape(n, 784)


def mirror(x):
    """Edit: half of the photos flipped left to right."""
    img = x.view(-1, 28, 28)
    flipped = torch.rand(len(x), 1, 1) < 0.5
    return torch.where(flipped, img.flip(2), img).reshape(-1, 784)


set_seed(SEED)
show_row(x_val[:8], "eight products of the catalogue")
show_row(shift(x_val[:8]), "shift: moved by up to 3 pixels")

### 1.2 ✏️ · The lighting edit

The second edit is nuisance B of last Monday's checkpoint 4: the photo is lit
differently. Each photo keeps only a share of its ink, between `lo` (25 %) and
100 %: multiply all its pixels by **one random factor**.

> **Watch out.** One factor **per photo**: a tensor of shape `(n, 1)`. Not one
> for the whole batch, and not one per pixel. Nothing crashes if you get it
> wrong.

`torch.empty(n, 1).uniform_(lo, 1.0)` draws `n` such factors.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     factor: one random number per photo, between lo and 1, shape (n, 1)
#     return the photos multiplied by their factor
# ──────────────────────────────────────────────────────────────────────
def relight(x, lo=0.25):
    """Edit: each photo keeps between lo and 100 % of its ink - its own factor."""
    ...          # <- replace this line


set_seed(SEED)
kept = relight(x_val[:5]).sum(1) / x_val[:5].sum(1)
print("ink kept by five photos:", [f"{k:.2f}" for k in kept.tolist()])
print("expect five different numbers between 0.25 and 1.00")

### 1.3 ✏️ · Two shots

A *list of edits* is a function that applies them one after the other. Ours,
for now: move, then relight.

Two shots of a batch of products = the list of edits applied **twice**. The
random numbers are drawn again the second time, so the two shots differ.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 or 2 lines
#     return two things: edit(x), and edit(x) again
# ──────────────────────────────────────────────────────────────────────
def standard(x):
    """Today's list of edits: move the product, then light it differently."""
    return relight(shift(x))


def two_shots(x, edit):
    """Shot A and shot B of every product of x: the same list of edits, drawn twice."""
    ...          # <- replace this line


set_seed(SEED)
shot_a, shot_b = two_shots(x_val[:8], standard)
show_row(x_val[:8], "the products")
show_row(shot_a, "shot A")
show_row(shot_b, "shot B")

### 1.4 ▶ · The customers' photos, and why pixels cannot pair them

The test of the day. The **catalogue** is the 6,000 held-out products. A
**customer** photographs 1,000 of them again: off-centre by up to 3 pixels, and
badly lit. Product *i* of `reshot` is product *i* of `catalogue`.

For each customer's photo the cell sorts the 6,000 catalogue photos from the
closest to the furthest, and prints the **rank of the right product**: 1 would
mean "the closest photo is the right one". It does so twice: comparing pixels,
and comparing last Monday's codes.

In [ ]:
# Given - read it, run it, move on.
catalogue = x_val                                    # 6,000 products
set_seed(SEED + 1)
reshot = shift(x_val[:1000]) * torch.empty(1000, 1).uniform_(0.25, 1.0)

show_row(catalogue[:8], "in the catalogue")
show_row(reshot[:8], "the customer's photo of the same product")


def rank_of_the_right_product(queries, cat):
    """For each query: 1 + how many catalogue rows are closer than its own product."""
    d = torch.cdist(queries, cat)
    own = d[torch.arange(len(queries)), torch.arange(len(queries))]
    return (d < own[:, None]).sum(1) + 1


r_pix = rank_of_the_right_product(reshot, catalogue)
r_ae = rank_of_the_right_product(codes(ae, reshot), codes(ae, catalogue))
print(f"comparing pixels            : median rank {int(r_pix.median()):>5} of 6000")
print(f"comparing last Monday's codes: median rank {int(r_ae.median()):>5} of 6000")

### ❓ Checkpoint 1

1. In 1.2, how much of its ink does each of the five photos keep? Are the five
   numbers different from each other?
2. In 1.3, look at shot A and shot B of the same product. What changed between
   the two, and what did not? Could **you** pair the shots A with the shots B?
3. In 1.4, at what rank does the right product come (median), comparing pixels?
   Comparing last Monday's codes? Out of how many?

**⭐ Going further - optional**

4. An AI's first draft of an edit often draws **one** random number for the
   whole batch - `torch.empty(1).uniform_(lo, 1.0)` in `relight`, or a single
   offset in `shift`. Every photo of shot A then gets the same edit, and every
   photo of shot B another one. Nothing crashes. In one step of training, how
   many different pairs of moves (shot A, shot B) does the network see when
   the move is drawn once per batch - and how many when each photo draws its
   own? *(Measured on the reference run, `found again` of checkpoint 3: 90 %
   with every edit drawn per photo, 79 % with the move drawn once per batch,
   89 % with the lighting drawn once per batch.)*

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 2 - The matching game

**~20 minutes.** Two cells to write.

The game, on a batch of 256 products: each shot A must pick **its own** shot B
among the 256 shots B. Two steps: compare every shot A with every shot B (the
**grid**), then score the 256 choices (the **loss**).

### 2.1 ✏️ · The grid

How alike two codes are is one number between −1 and 1: +1 when they point in
the same direction, 0 at right angles, −1 when they are opposite. Their length
does not count.

Two steps: give every code a length of 1 (`F.normalize(t, dim=1)`), then
multiply the two tables: `a @ b.T`. Row *i*, column *j* of the result: how
alike code *i* of `a` is to code *j* of `b`.

The check below uses the four arrows of the slide.

*(Papers call this number the cosine similarity.)*

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     give every code of a a length of 1: F.normalize(a, dim=1)
#     the same for b
#     return a @ b.T
# ──────────────────────────────────────────────────────────────────────
def grid(a, b):
    """How alike every code of a is to every code of b. Shape (len(a), len(b))."""
    ...          # <- replace this line


one = torch.tensor([[3.0, 4.0]])
four = torch.tensor([[6.0, 8.0], [4.0, 3.0], [-4.0, 3.0], [-3.0, -4.0]])
print("yours :", [f"{v:+.2f}" for v in grid(one, four)[0].tolist()])
print("slide :", ["+1.00", "+0.96", "+0.00", "-1.00"])

### 2.2 ✏️ · The loss

Each **row** of the grid is a multiple-choice question: "which of these shots B
is mine?". The right answer of row *i* is column *i*. That is exactly what a
classifier is trained on, so the loss is the classifier's loss,
`F.cross_entropy(scores, right_answers)`, where:

- the scores are the grid **divided by the temperature**;
- the right answers are 0, 1, 2, …: `torch.arange(len(a))`.

The shots B ask the same question about the shots A: play the game on the
**columns** too (the grid turned round: `g.T`) and average the two losses.

The check uses the toy example of the slide: four products, whose shots B are
"turned" by 30 degrees.

*(Papers call this loss InfoNCE, or NT-Xent.)*

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     g: the grid of a and b, divided by the temperature
#     target: the right answers, torch.arange(len(a))
#     return the average of F.cross_entropy(g, target) and F.cross_entropy(g.T, target)
# ──────────────────────────────────────────────────────────────────────
def matching_loss(a, b, temperature=0.1):
    """The matching game: row i of the grid must pick column i, and the other way round."""
    ...          # <- replace this line


angle = torch.deg2rad(torch.tensor([0.0, 90.0, 180.0, 270.0]))
toy_a = torch.stack([angle.cos(), angle.sin()], 1)                              # four codes
toy_b = torch.stack([(angle + torch.pi / 6).cos(), (angle + torch.pi / 6).sin()], 1)
print("first row of the grid:", [f"{v:+.2f}" for v in grid(toy_a, toy_b)[0].tolist()])
for t, slide in ((1.0, 0.754), (0.5, 0.456), (0.1, 0.025)):
    print(f"temperature {t:<4g}  yours {matching_loss(toy_a, toy_b, t).item():.3f}   slide {slide:.3f}")

### 2.3 · The network and the training loop

Given, and you will not touch them again. The network is **last Monday's
encoder and nothing else**: there is no decoder any more. The loop is last
Monday's loop, with two shots and `matching_loss` in place of the squared
error. The last, incomplete batch of each epoch is skipped, so that the grid is
always 256 × 256.

In [ ]:
# Given - read it, run it, move on.
class Encoder(nn.Module):
    """Last Monday's encoder and nothing else: 784 -> 256 -> d. No decoder."""

    def __init__(self, d=32, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, d))

    def forward(self, x):
        return self.net(x)


def train(model, edit, *, loss_fn=None, temperature=0.1, epochs=8, batch=256, lr=1e-3,
          verbose=True):
    """Play the game on x_train; return the average loss of each epoch."""
    loss_fn = loss_fn or matching_loss
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        perm = torch.randperm(len(x_train))
        total, steps = 0.0, 0
        for i in range(0, len(x_train) - batch + 1, batch):
            photos_a, photos_b = two_shots(x_train[perm[i:i + batch]], edit)
            a, b = model(photos_a), model(photos_b)       # the same encoder, twice
            loss = loss_fn(a, b, temperature)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); steps += 1
        history.append(total / steps)
        if verbose:
            print(f"  epoch {epoch + 1:>2}/{epochs}   loss = {history[-1]:.3f}")
    return history

### 2.4 ▶ · Play it: 8 epochs, a batch of 256, temperature 0.1

A few seconds on the reference machine. For scale: answering **at random**
among 256 shots gives a loss of log(256) = **5.55**.

In [ ]:
# Given - read it, run it, move on.
set_seed(SEED)
model = Encoder()
t0 = time.time()
hist = train(model, standard)
print(f"trained in {time.time() - t0:.1f}s")

### ❓ Checkpoint 2

1. Do your four numbers in 2.1 match the slide? The first arrow, (6, 8), is
   twice as long as (3, 4): what is their score, and does the length count?
2. In 2.2, do your three losses match the slide? The grid and the right answers
   are the same three times: at which temperature is the loss the lowest?
3. In 2.4, what is the loss after the first epoch, and after the last? Compare
   both with 5.55, the loss of an answer at random among 256.

**⭐ Going further - optional**

4. `exp(-loss)` is, roughly, the share of its choice that a shot gives to the
   right answer. What share does your last epoch reach? Among how many possible
   answers?

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 3 - Did it work?

**~15 minutes.** One cell to write.

### 3.1 ✏️ · Found again

The score of the day. Take the 1,000 customers' photos of 1.4, compare each
with the 6,000 products of the catalogue, and keep the most alike. **Found
again** = the share of customers' photos whose most alike catalogue product is
the right one.

`codes(enc, photos)` gives the codes of some photos with any encoder.
Row *i* of the queries is product *i* of the catalogue, so the right answers
are again 0, 1, 2, …

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     g: the grid between the codes of the queries and the codes of the catalogue
#     nearest: for each row of g, the column of the largest value (argmax along dim 1)
# ──────────────────────────────────────────────────────────────────────
def found_again(enc, queries):
    """Share of the queries whose most alike catalogue code is the same product."""
    ...          # <- replace this line
    return float((nearest == torch.arange(len(queries))).float().mean())


print(f"comparing pixels    found again: {found_again(nn.Identity(), reshot):.3f}")
print(f"last Monday's code  found again: {found_again(ae, reshot):.3f}")
print(f"today's code        found again: {found_again(model, reshot):.3f}")

### 3.2 ▶ · Look at four searches

Left: the customer's photo, and the product to find. Right: the six most alike
products of the catalogue, framed when it is the right one.

In [ ]:
# Given - read it, run it, move on.
def show_search(enc, title, queries=(0, 1, 2, 3)):
    top = grid(codes(enc, reshot), codes(enc, catalogue)).topk(6, dim=1).indices
    fig, axes = plt.subplots(len(queries), 8, figsize=(8.8, 1.15 * len(queries)))
    for r, q in enumerate(queries):
        shown = [reshot[q], catalogue[q]] + [catalogue[j] for j in top[q]]
        for c, img in enumerate(shown):
            ax = axes[r][c]
            ax.imshow(1 - img.reshape(28, 28).numpy(), cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            hit = c >= 2 and int(top[q, c - 2]) == q
            for sp in ax.spines.values():
                sp.set_visible(True)
                sp.set_color(PALETTE["teal"] if hit or c == 1 else PALETTE["accent"] if c == 0
                             else PALETTE["rule"])
                sp.set_linewidth(3 if hit else 1.5)
    axes[0][0].set_title("customer", fontsize=9); axes[0][1].set_title("to find", fontsize=9)
    axes[0][4].set_title(title, fontsize=10)
    plt.show()


first = [int(np.where(y_val[:1000] == k)[0][0]) for k in (7, 4, 3, 8)]   # sneaker, coat, dress, bag
show_search(ae, "the six most alike, for last Monday's code", first)
show_search(model, "the six most alike, for today's code", first)

### 3.3 ▶ · And last Monday's two numbers?

`knn` and `probe` ask something else: do the codes say what **kind** of product
it is? Nothing in the matching game asks for that.

In [ ]:
# Given - read it, run it, move on.
rep = evaluate_representation(codes(model, x_val).numpy(), y_val, name="today", verbose=False)
print(f"last Monday's code   knn {ae_rep.knn_accuracy:.3f}   probe {ae_rep.probe_accuracy:.3f}")
print(f"today's code         knn {rep.knn_accuracy:.3f}   probe {rep.probe_accuracy:.3f}")

### 3.4 ▶ · The trap, on purpose

The line everyone writes first for the loss: multiply the two tables of codes,
and call the classifier's loss. **No normalisation, no temperature.** It does
not crash. Same network, same edits, same 8 epochs.

In [ ]:
# Given - read it, run it, move on.
def first_draft_loss(a, b, temperature=None):
    return F.cross_entropy(a @ b.T, torch.arange(len(a)))     # the trap


set_seed(SEED)
trapped = Encoder()
hist_trap = train(trapped, standard, loss_fn=first_draft_loss, verbose=False)
rep_trap = evaluate_representation(codes(trapped, x_val).numpy(), y_val, name="trap", verbose=False)

print(f"first draft   loss {hist_trap[0]:.2f} -> {hist_trap[-1]:.2f}   knn {rep_trap.knn_accuracy:.3f}"
      f"   found again {found_again(trapped, reshot):.3f}")
print(f"yours         loss {hist[0]:.2f} -> {hist[-1]:.2f}   knn {rep.knn_accuracy:.3f}"
      f"   found again {found_again(model, reshot):.3f}")

### ❓ Checkpoint 3

1. Write down the three scores of 3.1. How many of the 1,000 customers' photos
   does last Monday's code find? And today's?
2. In 3.3, is today's code better or worse than last Monday's at saying the
   **kind** of product? By how much?
3. In 3.4, does the loss of the first draft go down? Is its `knn` different
   from yours? And its `found again`? Which of the three numbers reveals the
   bug?

**⭐ Going further - optional**

4. The customers' photos were made with the same two edits as the shots you
   trained on. What does that say about the score of today's code - and what
   would you expect on photos that differ in **another** way, say mirrored?
   *(Checkpoint 6 measures it.)*

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 4 - The edits are the instruction

**~20 minutes. The most important part of the lab.** One cell to write.

Each edit in the list is a declaration: *this does not change the product*. The
network obeys: what you declare irrelevant, the code learns to ignore. You will
train four encoders that differ **only** in their list of edits, and ask each
code three questions.

### 4.1 ▶ · Three questions we can ask a code

Each question is a small test set with a yes/no answer. A simple classifier is
trained to read the answer **from the codes**; `can_tell` returns its share of
right answers on held-out photos. **1.0**: the code tells perfectly. **0.5**:
the code has forgotten.

| Question | The test set |
|---|---|
| `faded photo?` | the 6,000 held-out photos; half of them badly lit (they keep 25 to 55 % of their ink) |
| `dark garment?` | the same photos, untouched: is this product darker than most products **of its kind**? |
| `turned shoe?` | the sandals, sneakers and boots; half of them flipped left to right |

The first is about the **photo**. The other two are about the **product**.

In [ ]:
# Given - read it, run it, move on.
g_ = torch.Generator().manual_seed(SEED + 3)
faded = torch.rand(6000, generator=g_) < 0.5
factor = torch.where(faded, 0.25 + 0.30 * torch.rand(6000, generator=g_),
                     0.70 + 0.30 * torch.rand(6000, generator=g_))

ink = np.array([p[p > 0.1].mean() for p in x_val.numpy()])    # how dark the garment itself is
dark = np.zeros(6000, dtype=int)
for c in range(10):
    s = y_val == c
    dark[s] = ink[s] > np.median(ink[s])                       # darker than most of its kind

is_shoe = torch.from_numpy(np.isin(y_val, (5, 7, 9)))          # sandal, sneaker, ankle boot
g_ = torch.Generator().manual_seed(SEED + 2)
turned = torch.rand(int(is_shoe.sum()), generator=g_) < 0.5
shoes = x_val[is_shoe].view(-1, 28, 28)
shoes = torch.where(turned[:, None, None], shoes.flip(2), shoes).reshape(-1, 784)

QUESTIONS = {
    "faded photo?": (x_val * factor[:, None], faded.numpy().astype(int)),
    "dark garment?": (x_val, dark),
    "turned shoe?": (shoes, turned.numpy().astype(int)),
}


def can_tell(enc, photos, answer):
    """Can a simple classifier read `answer` from the codes? 0.5 = it cannot."""
    z = codes(enc, photos).numpy()
    z = (z - z.mean(0)) / (z.std(0) + 1e-8)
    n = len(z) * 2 // 3
    clf = LogisticRegression(max_iter=500).fit(z[:n], answer[:n])
    return float(clf.score(z[n:], answer[n:]))


for question, (photos, answer) in QUESTIONS.items():
    show_row(photos, question, labels=["yes" if a else "no" for a in answer[:8]])
    print(f"{question:<14s} last Monday's code: {can_tell(ae, photos, answer):.3f}")

### 4.2 ✏️ · One row of the table

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     train m with this list of edits (verbose=False)
#     rep   = evaluate_representation(codes(m, x_val).numpy(), y_val, name=name, verbose=False)
#     found = found_again of m on the customers' photos (reshot)
# ──────────────────────────────────────────────────────────────────────
def policy_row(name, edit):
    """Train an encoder with this list of edits, measure it. Returns (row, model)."""
    set_seed(SEED)
    m = Encoder()
    ...          # <- replace this line
    row = {"edits": name, "found again": found, "knn": rep.knn_accuracy}
    for question, (photos, answer) in QUESTIONS.items():
        row[question] = can_tell(m, photos, answer)
    return row, m


row, _ = policy_row("move + light", standard)
print(row)          # sanity check: the numbers of checkpoint 3

### 4.3 ▶ · Four lists of edits, about fifteen seconds

In [ ]:
# Given - read it, run it, move on.
EDITS = {
    "move": lambda x: shift(x),
    "light": lambda x: relight(x),
    "move + light": lambda x: relight(shift(x)),
    "move + light + mirror": lambda x: relight(mirror(shift(x))),
}

rows, policy_models = [], {}
for name, edit in EDITS.items():
    row, policy_models[name] = policy_row(name, edit)
    rows.append(row)

last_monday = {"edits": "last Monday's AE", "found again": found_again(ae, reshot),
               "knn": ae_rep.knn_accuracy}
for question, (photos, answer) in QUESTIONS.items():
    last_monday[question] = can_tell(ae, photos, answer)

table = pd.DataFrame([last_monday] + rows).set_index("edits")
table.round(3)

### 4.4 ▶ · How strong? Move by 1, 3, 8 or 14 pixels

The same list, *move + light*, with a weaker and two stronger moves. At 14
pixels, half of the product can leave the frame.

In [ ]:
# Given - read it, run it, move on.
strength = [{"move by up to": "3 px", "found again": table.loc["move + light", "found again"],
             "knn": table.loc["move + light", "knn"]}]
for px in (1, 8, 14):
    row, _ = policy_row(f"{px} px", lambda x, px=px: relight(shift(x, px)))
    strength.append({"move by up to": f"{px} px", "found again": row["found again"],
                     "knn": row["knn"]})

set_seed(SEED)
show_row(shift(x_val[:8], 14), "moved by up to 14 pixels")
pd.DataFrame(strength).set_index("move by up to").loc[["1 px", "3 px", "8 px", "14 px"]].round(3)

### ❓ Checkpoint 4 - the questions that matter most today

`found again` and `knn` are the two scores of checkpoint 3. The three other
columns say what the code can still tell: 0.5 means forgotten.

1. Compare *move* with *move + light*: which column falls when the lighting
   edit joins the list? Compare *move + light* with *move + light + mirror*:
   which column falls now?
2. The customers' photos are moved **and** badly lit. What does *move* alone
   score on `found again`? And *light* alone? Which list finds them best?
3. `faded photo?` is about the photo; `dark garment?` is about the product.
   When you declare the lighting irrelevant, what happens to **both**? A
   customer asks for "the same one, in a darker shade": which list of edits can
   still serve them, and what does it cost on `found again`?

**⭐ Going further - optional**

4. In 4.4, a move of 1 pixel and a move of 14 pixels both find fewer photos
   than 3 pixels. The two reasons are different: which is which?
5. *light* alone has a lower `knn` than last Monday's autoencoder, and finds
   almost nothing. Its loss at the end of training is about 0.07: what does
   that say about how hard its game was?

*Your answers:*

1.
2.
3.
⭐ 4.
⭐ 5.

---

# Checkpoint 5 - Left alone, the network cheats

**~10 minutes.** One cell to write, one line.

The game has two halves: your own shot B must be alike (the pull), and **more
alike than everybody else's** (the other products of the batch). What if we
keep only the first half?

### 5.1 ✏️ · The pull alone

The loss: minus the average of how alike each shot A is to **its own** shot B.
Those 256 numbers are the diagonal of the grid: `grid(a, b).diag()`.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 line
#     return minus the average of the diagonal of grid(a, b)
# ──────────────────────────────────────────────────────────────────────
def pull_only(a, b, temperature=None):
    """Only the first instruction: the two shots of a product must be alike."""
    ...          # <- replace this line


set_seed(SEED)
lazy = Encoder()
hist_lazy = train(lazy, standard, loss_fn=pull_only, verbose=False)
print(f"loss after the first epoch {hist_lazy[0]:.3f}, after the last {hist_lazy[-1]:.3f}"
      f"   (the best possible value is -1)")

### 5.2 ▶ · What it learned

How alike are two **different** products, on average? And the two scores.

In [ ]:
# Given - read it, run it, move on.
def alike_on_average(enc):
    """How alike two different products are, on average (1,000 x 1,000 pairs)."""
    return float(grid(codes(enc, x_val[:1000]), codes(enc, x_val[1000:2000])).mean())


rep_lazy = evaluate_representation(codes(lazy, x_val).numpy(), y_val, name="pull", verbose=False)
for name, enc, r in (("last Monday's code", ae, ae_rep), ("the matching game", model, rep),
                     ("the pull alone", lazy, rep_lazy)):
    print(f"{name:<20s} two different products alike: {alike_on_average(enc):+.3f}   "
          f"found again {found_again(enc, reshot):.3f}   knn {r.knn_accuracy:.3f}")

### ❓ Checkpoint 5

1. What loss does the pull alone reach? Is it the best possible value?
2. How alike are two **different** products, on average, for each of the three
   codes?
3. What are `found again` and `knn` for the pull alone? Which half of the
   matching game was preventing this?

**⭐ Going further - optional**

4. For the matching game, two different products are alike at about 0.0 on
   average. With 32 numbers per code, what does a score of 0 mean for two
   codes - and why is that the best use of the space?

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 6 - The challenge: the customers use their phone

**Open-ended. Most of you will not get here, and that is fine.**

A harder set of customers' photos, `phone`: further off-centre (up to **4**
pixels), badly lit, and **mirrored half of the time** (the front camera).

> Build **one** encoder with the highest **`found again` on the phone photos**,
> while keeping **`knn` at 0.79 or more**, with a code of **at most 32
> numbers** and **no more than 60 seconds of training**.

The reference is today's model: about **47 %** at 0.80 `knn`.

| Lever | Try |
|---|---|
| `px` | how far the move edit goes: 3, 4, 6… |
| `with_mirror` | add the mirror edit to the list |
| `lo` | how badly lit a shot can be (0.25 keeps at least 25 % of the ink) |
| `temperature` | 0.05, 0.1, 0.2… |
| `batch` | how many products play together: 64, 256, 1024 |
| `epochs`, `hidden` | length of training, width of the network |

**The rules.** Time only the training. Keep a row for **every** configuration,
including the losers. One change at a time.

### 6.1 ▶ · The harness, given

In [ ]:
# Given - read it, run it, move on.
set_seed(SEED + 7)
phone = mirror(shift(x_val[:1000], 4)) * torch.empty(1000, 1).uniform_(0.25, 1.0)
show_row(catalogue[:8], "in the catalogue")
show_row(phone[:8], "the same products, from a phone")

results = []


def attempt(name, *, px=3, with_mirror=False, lo=0.25, temperature=0.1, batch=256, epochs=8,
            hidden=HIDDEN, d=32):
    """Train under the clock with this list of edits; record the row."""
    def edit(x):
        x = shift(x, px)
        if with_mirror:
            x = mirror(x)
        return relight(x, lo)

    set_seed(SEED)
    m = Encoder(d, hidden)
    t0 = time.time()
    train(m, edit, temperature=temperature, batch=batch, epochs=epochs, verbose=False)
    secs = time.time() - t0
    knn = evaluate_representation(codes(m, x_val).numpy(), y_val, name=name,
                                  verbose=False).knn_accuracy
    score = found_again(m, phone)
    ok = "ok " if knn >= 0.79 else "OUT"
    results.append({"config": name, "seconds": round(secs, 1), "knn": knn,
                    "found again (phone)": score, "valid": knn >= 0.79})
    print(f"  {ok} {name:<40s} {secs:5.1f}s   knn={knn:.4f}   found again (phone)={score:.3f}")
    return m


print(f"last Monday's code: found again (phone) = {found_again(ae, phone):.3f}")
_ = attempt("reference: move 3 px + light")

### 6.2 ✏️ · Your attempts

One `attempt(...)` per configuration. Change **one thing** at a time.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about as many lines as you have ideas
#     one attempt(...) per configuration, one change at a time
# ──────────────────────────────────────────────────────────────────────
# _ = attempt("+ mirror", with_mirror=True)
# best = attempt("+ mirror, px=4", with_mirror=True, px=4)   # keep your best model in `best`
# ... keep going, and keep the losers

In [ ]:
# Given - read it, run it, move on.
pd.DataFrame(results).sort_values(["valid", "found again (phone)"], ascending=False)

### ❓ Checkpoint 6

1. Your best valid configuration, its `found again (phone)` and its `knn`.
2. **Which single lever mattered most?** Which two rows of your table prove it?
3. Did any configuration fall **out** (`knn` below 0.79)? Which lever pushed it
   out?

**⭐ Going further - optional**

4. Ask your best model the question `turned shoe?` of checkpoint 4
   (`can_tell(best, *QUESTIONS["turned shoe?"])`). What did you give up to find
   the mirrored photos? For which products of a real catalogue would that be a
   problem?

*Your answers:*

1.
2.
3.
⭐ 4.

---

## What you built today

- Two **shots** of every product, made with edits - no label, no second
  photographer.
- The **matching game** as code: a grid of "how alike", and the loss of a
  classifier whose right answers are the diagonal.
- A measured answer to the problem of the day: customers' photos **found
  again** go from about 5 % with last Monday's code to about 90 % - and last
  Monday's own number, `knn`, does not improve.
- The lesson that outlives the method: **the list of edits is the
  instruction**. What you declare irrelevant is forgotten, including things a
  customer may ask for.

**Tomorrow:** the second shot becomes a sentence. Same grid, same game - a
photo and its title instead of two photos - and you will search the catalogue
by typing words.